<a href="https://colab.research.google.com/github/emmanuelmassawe/Sentiment-Analysis-with-huggingface_transformers/blob/main/Sentiment_Analysis_with_huggingface_transformers_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -U transformers
!pip install -U kagglehub
!pip install -U kaggle
!pip install -U evaluate


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 147.4 MB/s  0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.22.1
    Uninstalling transformers-5.22.1:
      Successfully uninstalled transformers-5.22.1
  Attempting uninstall: kaggle
    Found existing installation: kaggle 2.0.2
    Uninstalling kaggle-2.0.2:
      Successfully uninstalled kaggle-2.0.2


## importing necessary file

In [2]:
#importing necessary libraries
import pandas as pd
import argparse

import numpy as np
import evaluate
from datasets import load_dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

In [3]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("kashishparmar02/social-media-sentiments-analysis-dataset")

print("Path to dataset files:", path)


100%|██████████| 50.9k/50.9k [00:00<00:00, 1.22MB/s]

Extracting files...
Path to dataset files: /root/.cache/kagglehub/datasets/kashishparmar02/social-media-sentiments-analysis-dataset/versions/3


In [4]:
import os

print(os.listdir(path))

['sentimentdataset.csv']


## loading the dataset

In [5]:
df =pd.read_csv(os.path.join(path,'sentimentdataset.csv'))

In [6]:
df.head(10)

,Unnamed: 0.1,Unnamed: 0,Text,Sentiment,Timestamp,User,Platform,Hashtags,Retweets,Likes,Country,Year,Month,Day,Hour
0,0,0,Enjoying a beautiful day at the park! ...,Positive,2023-01-15 12:30:00,User123,Twitter,#Nature #Park,15.0,30.0,USA,2023,1,15,12
1,1,1,Traffic was terrible this morning. ...,Negative,2023-01-15 08:45:00,CommuterX,Twitter,#Traffic #Morning,5.0,10.0,Canada,2023,1,15,8
2,2,2,Just finished an amazing workout! 💪 ...,Positive,2023-01-15 15:45:00,FitnessFan,Instagram,#Fitness #Workout,20.0,40.0,USA,2023,1,15,15
3,3,3,Excited about the upcoming weekend getaway! ...,Positive,2023-01-15 18:20:00,AdventureX,Facebook,#Travel #Adventure,8.0,15.0,UK,2023,1,15,18
4,4,4,Trying out a new recipe for dinner tonight. ...,Neutral,2023-01-15 19:55:00,ChefCook,Instagram,#Cooking #Food,12.0,25.0,Australia,2023,1,15,19
5,5,5,Feeling grateful for the little things in lif...,Positive,2023-01-16 09:10:00,GratitudeNow,Twitter,#Gratitude #PositiveVibes,25.0,50.0,India,2023,1,16,9
6,6,6,Rainy days call for cozy blankets and hot coc...,Positive,2023-01-16 14:45:00,RainyDays,Facebook,#RainyDays #Cozy,10.0,20.0,Canada,2023,1,16,14
7,7,7,The new movie release is a must-watch! ...,Positive,2023-01-16 19:30:00,MovieBuff,Instagram,#MovieNight #MustWatch,15.0,30.0,USA,2023,1,16,19
8,8,8,Political discussions heating up on the timel...,Negative,2023-01-17 08:00:00,DebateTalk,Twitter,#Politics #Debate,30.0,60.0,USA,2023,1,17,8
9,9,9,Missing summer vibes and beach days. ...,Neutral,2023-01-17 12:20:00,BeachLover,Facebook,#Summer #BeachDays,18.0,35.0,Australia,2023,1,17,12


In [7]:
df.shape

(732, 15)

## dropping unnecessary columns

In [8]:
df = df[["Text", "Sentiment"]].dropna()

print(df.head())
print(df["Sentiment"].value_counts())

                                                Text    Sentiment
0   Enjoying a beautiful day at the park!        ...   Positive  
1   Traffic was terrible this morning.           ...   Negative  
2   Just finished an amazing workout! 💪          ...   Positive  
3   Excited about the upcoming weekend getaway!  ...   Positive  
4   Trying out a new recipe for dinner tonight.  ...   Neutral   
Sentiment
Positive         44
Joy              42
Excitement       32
Neutral          14
Contentment      14
                 ..
Triumph           1
Heartwarming      1
Obstacle          1
Sympathy          1
Pressure          1
Name: count, Length: 279, dtype: int64


In [9]:
# Ondoa missing values
df = df.dropna(subset=["Text", "Sentiment"]).copy()

# Safisha labels
df["Sentiment"] = df["Sentiment"].astype(str).str.strip().str.lower()


In [10]:
positive_labels = {
    "positive", "joy", "excitement", "contentment", "happiness", "gratitude",
    "hopeful", "hope", "awe", "elation", "euphoria", "enthusiasm", "curiosity",
    "admiration", "pride", "proud", "love", "amusement", "adoration", "kindness",
    "serenity", "enjoyment", "fulfillment", "accomplishment", "inspiration",
    "confidence", "playful", "compassion", "tenderness", "thrill", "success",
    "celebration", "determination", "optimism", "zest", "triumph",
    "heartwarming", "sympathy", "empathy", "relief", "wonder", "inspired",
    "free-spirited", "radiance", "harmony", "creativity", "satisfaction",
    "romance", "friendship", "freedom", "grateful", "happy",
}

negative_labels = {
    "negative", "sadness", "anger", "fear", "disgust", "loss", "despair",
    "grief", "loneliness", "heartbreak", "betrayal", "bitterness", "jealousy",
    "resentment", "frustration", "boredom", "anxiety", "hate", "sorrow",
    "regret", "darkness", "devastated", "isolation", "desperation",
    "helplessness", "overwhelmed", "fearful", "bad", "sad", "melancholy",
    "envy", "shame", "guilt", "embarrassed", "disappointed", "disappointment",
    "suffering", "pain", "intimidation", "numbness", "exhaustion",
    "emotionalbreakdown", "obstacle", "pressure", "anguish", "apprehensive",
    "bitter", "conflicted", "sad", "lonely", "miserable", "fearful",
}

neutral_labels = {"neutral", "ambivalence", "indifference", "calmness", "acceptance"}

def map_label(x):
    if x in positive_labels:
        return "positive"
    if x in negative_labels:
        return "negative"
    if x in neutral_labels:
        return "neutral"
    return None   # haijulikani

df["mapped"] = df["Sentiment"].apply(map_label)

# Angalia lebo ambazo bado hazijawekwa kwenye kundi lolote
unmapped = df[df["mapped"].isna()]["Sentiment"].value_counts()
print("Lebo ambazo hazijapangwa:", len(unmapped))
print(unmapped.to_string())

print(df["mapped"].value_counts(dropna=False))

Lebo ambazo hazijapangwa: 92
Sentiment
nostalgia               11
confusion                8
surprise                 6
empowerment              5
frustrated               5
compassionate            4
reverence                4
arousal                  4
reflection               4
enchantment              4
desolation               4
empathetic               3
envious                  3
confident                3
kind                     3
jealous                  3
dismissive               3
anticipation             3
adventure                3
affection                2
coziness                 2
resilience               2
captivation              2
exploration              2
rejuvenation             2
whimsy                   2
yearning                 2
contemplation            2
tranquility              2
emotion                  2
mischievous              2
blessed                  1
joyfulreunion            1
bittersweet              1
overjoyed                1
motivation      

In [11]:
df_bin = df[df["mapped"].isin(["positive", "negative"])].copy()
df_bin["labels"] = df_bin["mapped"].map({"negative": 0, "positive": 1})
df = df_bin[["Text", "labels"]]

print(df["labels"].value_counts())
print(df.shape)

labels
1    360
0    162
Name: count, dtype: int64
(522, 2)


In [12]:
print(df.columns.tolist())

['Text', 'labels']


In [13]:
df.head(2)

,Text,labels
0,Enjoying a beautiful day at the park! ...,1
1,Traffic was terrible this morning. ...,0


In [14]:
df.head(2)

,Text,labels
0,Enjoying a beautiful day at the park! ...,1
1,Traffic was terrible this morning. ...,0


## performing train_test and split

In [15]:
df = df.rename(columns={"Sentiment": "labels"})

In [16]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["labels"]
)

print("Train:", len(train_df))
print("Test:", len(test_df))

Train: 417
Test: 105


## Convert kwenda Hugging Face Dataset

In [17]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(
    train_df[["Text", "labels"]],
    preserve_index=False
)

test_dataset = Dataset.from_pandas(
    test_df[["Text", "labels"]],
    preserve_index=False
)

print(train_dataset)
print(test_dataset)

Dataset({
    features: ['Text', 'labels'],
    num_rows: 417
})
Dataset({
    features: ['Text', 'labels'],
    num_rows: 105
})


## loading our model

In [18]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "distilbert-base-uncased"

In [19]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    #id2label=id2label,
    #label2id=label2id
)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [20]:
id2label = {
    0: "NEGATIVE",
    1: "POSITIVE"
}

label2id = {
    "NEGATIVE": 0,
    "POSITIVE": 1
}

model.config.id2label = id2label
model.config.label2id = label2id

In [21]:
def tokenize_function(examples):
    return tokenizer(
        examples["Text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

train_tokenized = train_dataset.map(
    tokenize_function,
    batched=True
)

test_tokenized = test_dataset.map(
    tokenize_function,
    batched=True
)

Map:   0%|          | 0/417 [00:00<?, ? examples/s]

Map:   0%|          | 0/105 [00:00<?, ? examples/s]

In [22]:
print(train_tokenized.column_names)
print(test_tokenized.column_names)

['Text', 'labels', 'input_ids', 'token_type_ids', 'attention_mask']
['Text', 'labels', 'input_ids', 'token_type_ids', 'attention_mask']


In [23]:
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
import torch

## EVALUATION METRIC

In [24]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="binary"
    )

    accuracy = accuracy_score(
        labels,
        predictions
    )

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

TRAINING CONFIGURATION

In [25]:
training_args = TrainingArguments(
    output_dir="/content/sentiment_model",

    eval_strategy="epoch",
    save_strategy="epoch",

    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,

    num_train_epochs=3,

    weight_decay=0.01,

    load_best_model_at_end=True,

    metric_for_best_model="f1",
    greater_is_better=True,

    #logging_dir="/content/logs",
    logging_steps=50,

    report_to="none"
)

In [26]:
from transformers import DataCollatorWithPadding

# Use a data collator to handle dynamic padding during batching
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=test_tokenized,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,No log,0.404205,0.695238,0.692308,1.000000,0.818182
2,0.461409,0.281946,0.904762,0.878049,1.000000,0.935065
3,0.461409,0.236777,0.952381,0.935065,1.000000,0.966443


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=81, training_loss=0.3651381657447344, metrics={'train_runtime': 29.6759, 'train_samples_per_second': 42.155, 'train_steps_per_second': 2.729, 'total_flos': 41429178929664.0, 'train_loss': 0.3651381657447344, 'epoch': 3.0})

## evaluating the model

In [27]:
results = trainer.evaluate()

print(results)

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.461409,0.236777,3,0.952381,0.935065,1.000000,0.966443


{'eval_loss': 0.23677659034729004, 'eval_accuracy': 0.9523809523809523, 'eval_precision': 0.935064935064935, 'eval_recall': 1.0, 'eval_f1': 0.9664429530201343}


## creating a new sentiment

In [28]:
def predict_sentiment(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    inputs = {
        key: value.to(model.device)
        for key, value in inputs.items()
    }

    with torch.no_grad():
        outputs = model(**inputs)

    probabilities = torch.softmax(
        outputs.logits,
        dim=-1
    )

    prediction = torch.argmax(
        probabilities,
        dim=-1
    ).item()

    confidence = probabilities[0][prediction].item()

    return {
        "sentiment": id2label[prediction],
        "confidence": confidence
    }

In [32]:
predict_sentiment(
    "your service was soo bad ")

{'sentiment': 'NEGATIVE', 'confidence': 0.5570667386054993}

In [33]:
predict_sentiment(
    "your service was soo nice")

{'sentiment': 'POSITIVE', 'confidence': 0.7299752235412598}